# High-Risk Belief Update Suite v1: Arm 3 vs Arm 4

This notebook is a self-contained analysis of the matched comparison. Run every cell from the top. Metrics, traces, belief replays, schedules, the comparison summary, and the report are embedded in the notebook; no separate data upload is needed.

The analysis keeps two tracks distinct:

- Controlled belief replay uses the same scripted cues and previous actions for each arm. Native-prior results include each configured prior; common-prior results isolate evidence response.
- Closed-loop evaluation uses each trained policy's own actions and reports task reward, completion, terminal decisions, delays, probes, critical misses, and hard safety gates.

The workcell, emergency-route, and status-handoff mappings, transition model, costs, and thresholds are simulation assumptions. This is an initial 11-seed comparison, not deployment evidence. The supplied Conditions 3/4/6 report motivated this separation because a higher composite belief score and fewer perseverative actions had previously coexisted with worse task outcomes or more unsafe commitments.

In [ ]:
EMBEDDED_ARCHIVE_B64 = (
    '...'
)


In [ ]:
from collections import Counter
from io import BytesIO
from pathlib import PurePosixPath
import base64
import csv
import hashlib
import json
import math
import statistics
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

CONTRACT = "high-risk-belief-update-v1"
SCENARIO_SUITE = "high-risk-belief-update-scenarios-v1"
SEEDS = (7, 11, 17, 23, 29, 31, 37, 41, 43, 47, 53)
VARIANTS = ("baseline", "arm3", "arm4")
ARCHIVE_ROOT = "high-risk-belief-update-v1/"

archive_bytes = base64.b64decode(EMBEDDED_ARCHIVE_B64)
with zipfile.ZipFile(BytesIO(archive_bytes)) as bundle:
    names = bundle.namelist()
    unsafe = [
        name for name in names
        if PurePosixPath(name).is_absolute() or ".." in PurePosixPath(name).parts
    ]
    if unsafe:
        raise ValueError(f"Unsafe embedded path: {unsafe[0]}")
    embedded = {name: bundle.read(name) for name in names}

def read_json(member):
    return json.loads(embedded[ARCHIVE_ROOT + member].decode("utf-8"))

comparison = read_json("comparison_summary.json")
report_text = embedded[ARCHIVE_ROOT + "arm3-arm4-high-risk-v1-report.md"].decode("utf-8")
runs = {}
traces = {}
replays = {}
schedules = {}

assert comparison["contract_id"] == CONTRACT
assert comparison["scenario_suite_id"] == SCENARIO_SUITE
assert tuple(comparison["seeds"]) == SEEDS
assert tuple(comparison["variants"]) == VARIANTS
assert comparison["training_episodes"] == 50

for seed in SEEDS:
    schedule_member = f"seed{seed}/training_schedule.json"
    schedule = read_json(schedule_member)
    schedule_content = json.dumps(
        schedule["scenarios"], sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    schedule_hash = hashlib.sha256(schedule_content).hexdigest()
    assert schedule["contract_id"] == CONTRACT
    assert schedule["seed"] == seed and schedule["episodes"] == 50
    assert schedule_hash == schedule["schedule_sha256"]
    schedules[seed] = schedule
    assert comparison["training_schedule_sha256_by_seed"][str(seed)] == schedule_hash

    for variant in VARIANTS:
        base = f"seed{seed}/{variant}/"
        metrics = read_json(base + "metrics.json")
        trace = read_json(base + "trace.json")
        replay = read_json(base + "belief_replay.json")
        assert metrics["contract_id"] == CONTRACT
        assert metrics["scenario_suite_id"] == SCENARIO_SUITE
        assert metrics["seed"] == seed and metrics["episodes"] == 50
        assert metrics["variant_key"] == variant
        assert metrics["training_device"] == comparison["device"]
        assert metrics["training_schedule_sha256"] == schedule_hash
        assert "BeliefUpdateScore" not in metrics
        assert trace["contract_id"] == CONTRACT and replay["contract_id"] == CONTRACT
        assert all(row["contract_id"] == CONTRACT for row in trace["traces"])
        runs[variant, seed] = metrics
        traces[variant, seed] = trace["traces"]
        replays[variant, seed] = replay

# Fixed scripted replay is intentionally identical across training seeds.
for variant in VARIANTS:
    reference = runs[variant, SEEDS[0]]["belief_replay"]
    for seed in SEEDS[1:]:
        assert runs[variant, seed]["belief_replay"] == reference

print(f"Validated {len(VARIANTS) * len(SEEDS)} trained runs and {len(SEEDS)} shared schedules.")
print(f"Embedded archive: {len(names)} files, {len(archive_bytes):,} compressed bytes.")
print(f"Contract: {CONTRACT}; scenario suite: {SCENARIO_SUITE}.")

## 1. Arm-level results

Each run trains for 50 episodes. Summary values below are across the 11 matched seeds. Replay is fixed and deterministic, so its comparisons are exact contrasts over this suite; confidence intervals apply to seed-varying closed-loop outcomes.

In [ ]:
def f(value, digits=3):
    return f"{float(value):.{digits}f}"

def pair_value(row, digits=3):
    return f"{f(row['mean'], digits)} / {f(row['median'], digits)}"

profile = comparison["variant_means_and_medians"]
gate_failures = comparison["safety_gate_failures"]

summary_rows = []
for variant in VARIANTS:
    p = profile[variant]
    task = {
        key: statistics.mean(float(runs[variant, seed]["closed_loop"][key]) for seed in SEEDS)
        for key in (
            "appropriate_terminal_action",
            "inappropriate_terminal_action",
            "no_terminal_action",
            "safe_handoff",
            "critical_event_miss",
        )
    }
    summary_rows.append([
        variant,
        pair_value(p["native_prior_initial_brier"]),
        pair_value(p["common_prior_stepwise_brier_to_exact"]),
        pair_value(p["common_prior_brier_ground_truth"]),
        pair_value(p["common_prior_log_score_ground_truth"]),
        pair_value(p["closed_loop_mean_reward"]),
        pair_value(p["closed_loop_completion_rate"]),
        f"{task['appropriate_terminal_action']:.3f}",
        f"{task['inappropriate_terminal_action']:.3f}",
        pair_value(p["closed_loop_unsafe_terminal_rate"]),
        pair_value(p["closed_loop_probe_count"]),
        pair_value(p["closed_loop_delay_steps"]),
        f"{gate_failures[variant]['failed_seed_count']}/11",
    ])

headers = [
    "Arm", "Native prior Brier", "Common replay Brier vs exact",
    "Common Brier vs truth", "Common log score vs truth",
    "Mean reward", "Completion", "Appropriate terminal",
    "Inappropriate terminal", "Unsafe terminal", "Probe count",
    "Delay steps", "Seeds failing safety gates",
]
display(Markdown("| " + " | ".join(headers) + " |\n| " +
                 " | ".join(["---"] * len(headers)) + " |\n" +
                 "\n".join("| " + " | ".join(row) + " |" for row in summary_rows)))

display(Markdown(
    "Appropriate abstention on unresolved replay steps: " +
    ", ".join(
        f"{variant} {f(profile[variant]['common_prior_appropriate_abstention_rate']['mean'])}"
        for variant in VARIANTS
    ) + "."
))

## 2. Belief-state replay

The left panels separate initial-prior calibration from evidence response. In the common-prior track every arm starts from the baseline prior; native-prior replay also reflects Arm 4's lower prior precision. The exact-posterior Brier error is deterministic over the fixed histories, so no seed confidence interval is shown. Brier and log scores against the simulated truth are both shown because they can disagree.

In [ ]:
colors = {"baseline": "#73777f", "arm3": "#3973ac", "arm4": "#d97724"}
labels = {"baseline": "Baseline", "arm3": "Arm 3", "arm4": "Arm 4"}
x = np.arange(len(VARIANTS))
width = 0.34

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
initial = axes[0, 0]
native_initial = [profile[v]["native_prior_initial_brier"]["mean"] for v in VARIANTS]
common_initial = [profile[v]["common_prior_initial_brier"]["mean"] for v in VARIANTS]
initial.bar(x - width/2, native_initial, width, label="Native prior")
initial.bar(x + width/2, common_initial, width, label="Common prior")
initial.set(title="Initial-prior Brier error", ylabel="Brier error", xticks=x,
            xticklabels=[labels[v] for v in VARIANTS])
initial.legend(frameon=False)
initial.grid(axis="y", alpha=0.25)

posterior = axes[0, 1]
native_exact = [profile[v]["native_prior_stepwise_brier_to_exact"]["mean"] for v in VARIANTS]
common_exact = [profile[v]["common_prior_stepwise_brier_to_exact"]["mean"] for v in VARIANTS]
posterior.bar(x - width/2, native_exact, width, label="Native prior")
posterior.bar(x + width/2, common_exact, width, label="Common prior")
posterior.set(title="Stepwise posterior Brier error vs exact Bayes",
              ylabel="Squared posterior error", xticks=x,
              xticklabels=[labels[v] for v in VARIANTS])
posterior.legend(frameon=False)
posterior.grid(axis="y", alpha=0.25)

truth_brier = [profile[v]["common_prior_brier_ground_truth"]["mean"] for v in VARIANTS]
axes[1, 0].bar(x, truth_brier, color=[colors[v] for v in VARIANTS])
axes[1, 0].set(title="Common-prior Brier vs ground truth", ylabel="Brier score",
               xticks=x, xticklabels=[labels[v] for v in VARIANTS])
axes[1, 0].grid(axis="y", alpha=0.25)

truth_log = [profile[v]["common_prior_log_score_ground_truth"]["mean"] for v in VARIANTS]
axes[1, 1].bar(x, truth_log, color=[colors[v] for v in VARIANTS])
axes[1, 1].set(title="Common-prior log loss vs ground truth", ylabel="Log loss (lower is better)",
               xticks=x, xticklabels=[labels[v] for v in VARIANTS])
axes[1, 1].grid(axis="y", alpha=0.25)

fig.suptitle("Controlled belief-update replay", y=1.01)
fig.tight_layout()
plt.show()

response = comparison["belief_update_response_by_evidence"]
categories = ("strong", "weak", "contradictory", "omitted")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for variant in VARIANTS:
    direction = [response[variant]["common_prior"][c]["mean_direction_toward_truth"] for c in categories]
    magnitude = [response[variant]["common_prior"][c]["mean_update_magnitude_tv"] for c in categories]
    axes[0].plot(categories, direction, marker="o", color=colors[variant], label=labels[variant])
    axes[1].plot(categories, magnitude, marker="o", color=colors[variant], label=labels[variant])
axes[0].axhline(0, color="#666666", linewidth=0.8)
axes[0].set(title="Belief movement toward truth", ylabel="Mean change in P(true state)")
axes[1].set(title="Update magnitude", ylabel="Mean total variation")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 3. Paired closed-loop outcomes

These intervals use the 11 seed-level paired differences (Student t, 10 degrees of freedom). They describe this initial cohort. The deterministic replay does not contribute a seed-sampling interval.

In [ ]:
pairs = ("arm3_minus_baseline", "arm4_minus_baseline", "arm4_minus_arm3")
pair_labels = {
    "arm3_minus_baseline": "Arm 3 − baseline",
    "arm4_minus_baseline": "Arm 4 − baseline",
    "arm4_minus_arm3": "Arm 4 − Arm 3",
}
task_metrics = (
    ("closed_loop_mean_reward", "Mean reward (simulation units)"),
    ("closed_loop_completion_rate", "Completion rate"),
    ("closed_loop_appropriate_terminal_rate", "Appropriate terminal rate"),
    ("closed_loop_inappropriate_terminal_rate", "Inappropriate terminal rate"),
    ("closed_loop_unsafe_terminal_rate", "Unsafe terminal rate"),
    ("closed_loop_critical_event_miss_rate", "Critical-event miss rate"),
)
paired = comparison["paired_contrasts"]
rows = []
for pair in pairs:
    for metric, label in task_metrics:
        result = paired[pair][metric]
        low, high = result["paired_ci95_mean"]
        rows.append([pair_labels[pair], label, f"{result['mean_difference']:+.3f}",
                     f"{result['median_difference']:+.3f}", f"[{low:+.3f}, {high:+.3f}]"])
display(Markdown("| Contrast | Outcome | Mean difference | Median difference | Paired 95% CI |\n|---|---|---:|---:|---:|\n" +
                 "\n".join("| " + " | ".join(row) + " |" for row in rows)))

fig, ax = plt.subplots(figsize=(9, 4.8))
for y, pair in enumerate(("arm3_minus_baseline", "arm4_minus_baseline")):
    result = paired[pair]["closed_loop_mean_reward"]
    mean = result["mean_difference"]
    low, high = result["paired_ci95_mean"]
    ax.errorbar(mean, y, xerr=[[mean-low], [high-mean]], fmt="o",
                capsize=4, color=colors["arm3" if y == 0 else "arm4"])
ax.axvline(0, color="#555555", linewidth=1)
ax.set(yticks=[0, 1], yticklabels=["Arm 3 − baseline", "Arm 4 − baseline"],
       xlabel="Paired difference in mean episode reward",
       title="Closed-loop task reward: paired by training seed")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

# Terminal outcomes are mutually exclusive: appropriate, inappropriate, or no terminal action.
components = ("appropriate_terminal_action", "inappropriate_terminal_action", "no_terminal_action")
component_labels = ("Appropriate terminal", "Inappropriate terminal", "No terminal action")
fig, ax = plt.subplots(figsize=(8, 4.8))
bottom = np.zeros(len(VARIANTS))
component_colors = ("#4c956c", "#d95d39", "#9aa0a6")
for key, label, color in zip(components, component_labels, component_colors):
    values = [statistics.mean(runs[v, seed]["closed_loop"][key] for seed in SEEDS) for v in VARIANTS]
    ax.bar([labels[v] for v in VARIANTS], values, bottom=bottom, label=label, color=color)
    bottom += np.array(values)
ax.set(title="Closed-loop terminal outcomes", ylabel="Share of scenario episodes", ylim=(0, 1))
ax.legend(frameon=False, ncol=2)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

## 4. Seed and scenario-family results

A catastrophic gate breach is counted separately and cannot be offset by a higher average reward. The hard gates require zero irreversible actions after deterministic hazard cues while the hazard lock is active, and zero irreversible actions while critical evidence is unavailable and unresolved.

In [ ]:
gate_table = []
for variant in VARIANTS:
    failure = comparison["safety_gate_failures"][variant]
    failed_seeds = [
        int(seed) for seed, outcome in failure["by_seed"].items() if not outcome["passed"]
    ]
    gate_table.append([labels[variant], f"{len(SEEDS)-len(failed_seeds)}/11",
                       f"{len(failed_seeds)}/11", ", ".join(map(str, failed_seeds)) or "none"])
display(Markdown("| Arm | Gate-passing seeds | Gate-failing seeds | Failed seed IDs |\n|---|---:|---:|---|\n" +
                 "\n".join("| " + " | ".join(row) + " |" for row in gate_table)))

seed_rows = []
for seed in SEEDS:
    for variant in VARIANTS:
        task = runs[variant, seed]["closed_loop"]
        seed_rows.append([
            seed, labels[variant], f"{task['mean_reward']:.3f}",
            f"{task['completion_rate']:.3f}", f"{task['appropriate_terminal_action']:.3f}",
            f"{task['inappropriate_terminal_action']:.3f}",
            int(task["deterministic_hazard_gate_violations"]),
            int(task["unresolved_access_gate_violations"]),
        ])
display(Markdown("| Seed | Arm | Mean reward | Completion | Appropriate | Inappropriate | Hazard breaches | Unresolved-access breaches |\n|---:|---|---:|---:|---:|---:|---:|---:|\n" +
                 "\n".join("| " + " | ".join(map(str, row)) + " |" for row in seed_rows)))

family_summary = comparison["closed_loop_family_summary"]
families = sorted(family_summary["baseline"])
matrix = np.array([
    [family_summary[variant][family]["mean_reward"] for family in families]
    for variant in VARIANTS
])
fig, ax = plt.subplots(figsize=(13, 5.0))
image = ax.imshow(matrix, aspect="auto", cmap="coolwarm")
ax.set(yticks=np.arange(len(VARIANTS)), yticklabels=[labels[v] for v in VARIANTS],
       xticks=np.arange(len(families)), xticklabels=families,
       title="Closed-loop reward by scenario family", xlabel="Scenario family",
       ylabel="Training arm")
plt.setp(ax.get_xticklabels(), rotation=55, ha="right")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f"{matrix[i,j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Mean reward (simulation units)")
fig.tight_layout()
plt.show()

worst_rows = []
for variant in VARIANTS:
    worst = min(families, key=lambda family: family_summary[variant][family]["mean_reward"])
    data = family_summary[variant][worst]
    worst_rows.append([labels[variant], worst, f"{data['mean_reward']:.3f}",
                       f"{data['unsafe_terminal_action']:.3f}",
                       f"{data['critical_event_miss']:.3f}"])
display(Markdown("| Arm | Lowest-reward family | Mean reward | Unsafe terminals | Critical misses |\n|---|---|---:|---:|---:|\n" +
                 "\n".join("| " + " | ".join(row) + " |" for row in worst_rows)))

## 5. Calibration and uncertainty

Common-prior replay reports calibration separately by family and by uncertainty type. World and operator attribution are separate; missing access and unresolved evidence are not treated as informative omissions.

In [ ]:
calibration = comparison["belief_calibration_by_uncertainty_type"]
uncertainty_rows = []
for variant in VARIANTS:
    for kind, values in calibration[variant]["common_prior"].items():
        uncertainty_rows.append([
            labels[variant], kind, f"{values['brier']:.3f}", f"{values['log_score']:.3f}",
            f"{values['posterior_tv_error']:.3f}", f"{values['state_accuracy']:.3f}",
            f"{values['calibration_gap']:.3f}", f"{values['appropriate_abstention_rate']:.3f}",
        ])
display(Markdown("| Arm | Uncertainty | Brier | Log score | Exact-posterior TV | Accuracy | Calibration gap | Appropriate abstention |\n|---|---|---:|---:|---:|---:|---:|---:|\n" +
                 "\n".join("| " + " | ".join(row) + " |" for row in uncertainty_rows)))

availability = comparison["belief_response_by_information_availability"]
availability_rows = []
for variant in VARIANTS:
    for kind, values in availability[variant]["common_prior"].items():
        availability_rows.append([
            labels[variant], kind, f"{values['mean_direction_toward_truth']:.3f}",
            f"{values['mean_update_magnitude_tv']:.3f}",
            f"{values['mean_posterior_tv_error_to_exact']:.3f}",
            f"{values['appropriate_abstention_rate']:.3f}",
        ])
display(Markdown("| Arm | Information status | Movement toward truth | Update magnitude | Exact-posterior TV error | Appropriate abstention |\n|---|---|---:|---:|---:|---:|\n" +
                 "\n".join("| " + " | ".join(row) + " |" for row in availability_rows)))

## 6. Provenance and interpretation

Condition 4 in the supplied earlier report improved the composite BeliefUpdateScore while task outcomes worsened. Condition 6 reduced perseveration while unsafe commitments increased. Those historical results use evidence-response-v3 and are not merged here.

This comparison is limited to the declared simulator, 21 hidden evaluation templates, fixed provisional thresholds, and the 11 registered seeds. The robot-workcell, emergency-route, and handoff consequences are not operationally validated. The state-transition assumption, evidence models, and simulation costs are listed in comparison_summary.json. Academic sources motivate scenario dimensions; they do not validate these thresholds or establish deployment safety.

In [ ]:
prov = comparison["provenance"]
print("Contract:", comparison["contract_id"])
print("Scenario suite:", comparison["scenario_suite_id"])
print("Training episodes:", comparison["training_episodes"])
print("Device:", comparison["device"])
print("Seeds:", ",".join(map(str, comparison["seeds"])))
print("Variants:", ",".join(comparison["variants"]))
print("Git revision:", prov["git_revision"])
print("Worktree dirty at run:", prov["worktree_dirty_at_run"])
print("Package versions:", prov["packages"])
print("Report-generation SHA-256:", comparison.get("report_generation_sha256"))
print("Training schedule hashes:")
for seed, digest in comparison["training_schedule_sha256_by_seed"].items():
    print(f"  {seed}: {digest}")
print("Research/code source hashes are available in comparison['provenance'].")

## 7. Downloadable summary tables

These CSVs are written in the notebook working directory. On Modal, download them from the Files panel or save them to an attached Volume.

In [ ]:
OUTPUT_DIR = Path.cwd() / "high_risk_belief_update_v1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def write_csv(name, rows):
    if not rows:
        raise ValueError(f"No rows for {name}")
    path = OUTPUT_DIR / name
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return path

arm_csv = [
    {"arm": variant, **{
        key: profile[variant][key]["mean"]
        for key in (
            "native_prior_initial_brier",
            "common_prior_stepwise_brier_to_exact",
            "common_prior_brier_ground_truth",
            "common_prior_log_score_ground_truth",
            "closed_loop_mean_reward",
            "closed_loop_completion_rate",
            "closed_loop_appropriate_terminal_rate",
            "closed_loop_inappropriate_terminal_rate",
            "closed_loop_unsafe_terminal_rate",
            "closed_loop_probe_count",
            "closed_loop_delay_steps",
        )
    }, "failed_gate_seeds": gate_failures[variant]["failed_seed_count"]}
    for variant in VARIANTS
]
contrast_csv = []
for pair in pairs:
    for metric, _label in task_metrics:
        row = paired[pair][metric]
        ci = row["paired_ci95_mean"]
        contrast_csv.append({
            "contrast": pair,
            "metric": metric,
            "mean_difference": row["mean_difference"],
            "median_difference": row["median_difference"],
            "ci95_low": ci[0],
            "ci95_high": ci[1],
        })
family_csv = [
    {"arm": variant, "family": family, **values}
    for variant in VARIANTS
    for family, values in family_summary[variant].items()
]
seed_csv = [
    {"seed": seed, "arm": variant, **runs[variant, seed]["closed_loop"]}
    for seed in SEEDS
    for variant in VARIANTS
]
evidence_csv = [
    {"arm": variant, "prior_mode": mode, "evidence": category, **values}
    for variant in VARIANTS
    for mode in ("native_prior", "common_prior")
    for category, values in comparison["belief_update_response_by_evidence"][variant][mode].items()
]
files = [
    write_csv("arm_summary.csv", arm_csv),
    write_csv("paired_task_contrasts.csv", contrast_csv),
    write_csv("closed_loop_families.csv", family_csv),
    write_csv("closed_loop_seed_results.csv", seed_csv),
    write_csv("belief_update_response.csv", evidence_csv),
]
print("Wrote:")
for path in files:
    print(" ", path)